# 03 Recommender design

This notebook measures how much of clicking depends on what is shown, fits a click model of user, card, slot and page depth, and evaluates two levers: the order of the cards within a page and the kind of cards on the first page. It then translates the levers into the share of inactive users, extra clicks and the size of a test.

Outputs: `15_*`, `21_*`, `23_*`, `24_*`, `27_*`. The slow steps are functions in `src/thesis/heavy/recommender.py`, read through `ensure(step)`.

## 1. Setup

In [1]:
# Libraries and shared project code
import json
import numpy as np
import pandas as pd
from IPython.display import display
from thesis.config import P, O, TAG, connect
from thesis.cache import ensure, read

## 2. Clicks explained by what the user is shown

In [2]:
# Exposure model (card x slot x page depth, pooled slots), expected clicks per snapshot, and the four click models with
# bootstrap intervals for the difference from the notebook-02 model (thesis.heavy.recommender.exposure)
E = ensure("03.exposure")["15_exposure"]
pd.DataFrame(E["models"]).T

[03.exposure] saved results loaded


,pseudo_r2,auc_any_click_20plus,r2_minus_step10_ci95,auc_minus_step10_ci95
exposure_only,0.0245,0.8053,"[-0.0726, -0.0621]","[-0.0138, 0.0]"
lightgbm_step10,0.0916,0.8121,"[0.0, 0.0]","[0.0, 0.0]"
lightgbm_plus_adjusted_history,0.0916,0.8122,"[-0.0002, 0.0002]","[-0.0004, 0.0004]"
user_x_exposure,0.0998,0.8277,"[0.0076, 0.0089]","[0.0145, 0.0167]"


In [3]:
# Exposure model factors and the correlation of expected with observed clicks
pd.Series({k: E[k] for k in ("exposure_model_slot_factors", "exposure_model_page_depth_factors",
                             "exposure_only_intercept", "corr_expected_vs_observed_clicks_next7_test")},
          name="value").to_frame()

,value
exposure_model_slot_factors,"[1.0, 0.828, 0.546, 0.434, 0.494, 0.394]"
exposure_model_page_depth_factors,"[1.0, 1.25, 1.169, 1.049, 0.864]"
exposure_only_intercept,0.3046
corr_expected_vs_observed_clicks_next7_test,0.8215


## 3. Click model

The chance of a click is the product of a user factor, a card factor, a slot factor and a page-depth factor. The model is fitted on training users' days 1 to 15, and its version is chosen on validation users.

In [4]:
# Nested click models fitted on training users, days 1-15; out-of-sample check on validation and test users, days 16-30;
# the model chosen on validation users; slot-factor intervals; re-ordering within pages
# (thesis.heavy.recommender.click_model)
RR = ensure("03.click_model")
res = RR["21_rerank"]
print("Model chosen on validation users:", res["rerank_within_pages"]["model_used"])
pd.concat({"validation_users": pd.DataFrame({k: v for k, v in res["model_choice_validation_users_days16_30"].items()
                                             if isinstance(v, dict)}).T,
           "test_users": pd.DataFrame({k: v for k, v in res["out_of_sample_check_test_days16_30"].items()
                                       if isinstance(v, dict)}).T})

[03.click_model] saved results loaded


Model chosen on validation users: plus_slot


auc  logloss  new_cards_auc  \
validation_users user_x_card                  0.6798  0.18732         0.7104   
                 plus_page_depth              0.6928  0.18620         0.7268   
                 plus_slot                    0.7148  0.18368         0.7364   
                 plus_slot_by_scroll_depth    0.7156  0.18455         0.7462   
                 plus_creator_topic_fallback  0.7165  0.18459         0.7421   
test_users       user_x_card                  0.6826  0.18861         0.7239   
                 plus_page_depth              0.6958  0.18752         0.7422   
                 plus_slot                    0.7170  0.18503         0.7486   
                 plus_slot_by_scroll_depth    0.7172  0.18588         0.7564   
                 plus_creator_topic_fallback  0.7180  0.18593         0.7521   

                                              new_cards_logloss  
validation_users user_x_card                            0.13252  
                 plus_page_depth                        0.13256  
                 plus_slot                              0.12855  
                 plus_slot_by_scroll_depth              0.12590  
                 plus_creator_topic_fallback            0.12566  
test_users       user_x_card                            0.13321  
                 plus_page_depth                        0.13315  
                 plus_slot                              0.12930  
                 plus_slot_by_scroll_depth              0.12661  
                 plus_creator_topic_fallback            0.12626

In [5]:
# Bounds: the click model is a rate model, so a predicted rate can exceed 1. How often does that happen for test users'
# impressions on days 16-30 (rates rescaled to the observed total, as in the evaluation, before any capping)?
bounds = RR["21_click_model_bounds"]
pd.Series({k: v for k, v in bounds.items() if not isinstance(v, dict)}, name="value").to_frame()

,value
model,plus_slot
test_impressions,3987537
share_rate_above_1,0.000003
share_rate_above_0.5,0.000269
max_rate,1.588
share_of_expected_clicks_on_rows_above_1,0.00007


## 4. Re-ordering with the click model

The same cards in the same slots, sorted by a score, with expected clicks from the click model.

In [6]:
# Expected clicks under each order and the inactive share after re-ordering
pd.Series({**res["rerank_within_pages"], **res["active_users"]}, name="value").to_frame()

,value
model_used,plus_slot
test_cards,3987537
pages,1593665
share_pages_2plus_cards,0.6869
rank_corr_logged_slot_vs_card_ctr_within_pages,-0.0303
expected_clicks_change_pct_platform_rate,2.56
expected_clicks_change_pct_platform_rate_ci95,"[2.53, 2.58]"
expected_clicks_change_pct_platform_rate_creator_aware,2.52
expected_clicks_change_pct_platform_rate_creator_aware_ci95,"[2.5, 2.54]"
expected_clicks_change_pct_click_model_appeal,4.03


In [7]:
# Click propensity of each slot relative to the first, by scroll-depth group (share of cards beyond the first page)
pd.DataFrame(np.reshape(res["slot_factors"], (-1, 6)), columns=[f"slot_{i + 1}" for i in range(6)],
             index=["first page only", "beyond it for under half of cards", "beyond it for half or more"] if len(res["slot_factors"]) == 18 else ["pooled"])

,slot_1,slot_2,slot_3,slot_4,slot_5,slot_6
pooled,1.0,0.826,0.545,0.433,0.492,0.39


## 5. Re-ordering in the logged clicks

Within pages of the same size, the order shown is almost unrelated to the cards' click rates, so the click rate of the card of each rank in each slot can be read from the logs. A sorted page gets the sum of the rates of rank 1 in slot 1, rank 2 in slot 2 and so on. The check covers days 16 to 30 and pages of two to six cards, with a random order as a placebo. A second version leaves new cards in their slot and sorts only the established ones.

In [8]:
# Observed gain from sorting each page by a score, with a random order as a check (thesis.heavy.recommender.observed)
OB = ensure("03.observed")
obs, balance, slot_cards = OB["21_observed_rerank"], OB["21_observed_balance"], OB["21_slot_cards"]
pd.DataFrame({(p, v): {n: obs[p][n][k] for n in ("platform_rate", "platform_rate_creator_aware", "random_order")}
              for p in ("test_users", "all_users")
              for v, k in (("feed order", "gain_pct"), ("slots by click rate", "gain_pct_slots_by_click_rate"))})

[03.observed] saved results loaded


test_users                      all_users  \
                            feed order slots by click rate feed order   
platform_rate                     4.33                1.16       4.47   
platform_rate_creator_aware       5.65                0.27       5.80   
random_order                      0.17                0.19       0.22   

                                                 
                            slots by click rate  
platform_rate                              1.17  
platform_rate_creator_aware                0.24  
random_order                               0.07

In [9]:
# Balance: within each rank, the cards' platform click rate by slot (pages of 4 cards)
balance[balance.k == 4].pivot(index="r", columns="s", values="mean_card_ctr").round(4)

s,0,1,2,3
r,,,,
0,0.0788,0.0696,0.0774,0.0774
1,0.0559,0.0556,0.0549,0.0573
2,0.0438,0.0426,0.0430,0.0416
3,0.0321,0.0228,0.0318,0.0311


In [10]:
# Pages of 4 cards: platform impressions of each card before the day, by its rank (0 = most clickable) and its slot.
# If the second slot serves to try out new cards, the least clickable cards placed there should have little history.
slot_cards.pivot(index="r", columns="s", values="median_prior_impressions").join(
    slot_cards.pivot(index="r", columns="s", values="share_under_100_impressions").round(3), rsuffix="_new_share")

s,0,1,2,3,0_new_share,1_new_share,2_new_share,3_new_share
r,,,,,,,,
0,73025.0,512.0,64710.0,66196.0,0.011,0.407,0.007,0.018
1,159022.0,607.0,67273.0,83949.0,0.014,0.402,0.009,0.026
2,202027.0,1215.0,94068.0,54964.0,0.015,0.256,0.010,0.021
3,76575.0,396066.0,62998.0,53168.0,0.014,0.049,0.008,0.017


In [11]:
# Established cards only: new cards keep their slot and the others are sorted, by the number of earlier
# impressions that makes a card established
pd.DataFrame({p: {k: v["gain_pct"] for k, v in obs[p]["established_cards_only"].items()}
              for p in ("test_users", "all_users")}).rename_axis("minimum earlier impressions")

,test_users,all_users
minimum earlier impressions,,
100,5.84,5.90
1000,5.35,5.37
10000,3.76,3.91


## 6. First-page content and return

Each user's days are compared with one another, with user and day fixed effects. A reverse check uses the six days before as the outcome, and the joint model is also estimated without last week's visits as a control.

In [12]:
# Card features per snapshot for validation and test users, the best first page buildable from each day's cards, and
# A: quality only, B: joint model (two-way fixed effects, user-clustered standard errors)
# (thesis.heavy.recommender.first_page)
FP = ensure("03.first_page")
res = FP["23_content_mix"]
FEAS = read("23_first_page_options.parquet")                             # used in section 7

# A: quality only
pd.concat({f: pd.DataFrame(res["A_quality_only"][f]).T for f in ("quality", "quality_all")})

[03.first_page] saved results loaded


coef       se  \
quality     return_next7                                0.01474  0.00043   
            visit_days_next7                            0.01175  0.00121   
            reverse_check_past6_visit_days             -0.01536  0.00124   
            return_next7_controlling_clicks_today       0.01437  0.00043   
            return_next7_without_previous_week_control  0.01892  0.00044   
quality_all return_next7                                0.01491  0.00042   
            visit_days_next7                            0.00754   0.0012   
            reverse_check_past6_visit_days             -0.02488  0.00122   
            return_next7_controlling_clicks_today        0.0144  0.00042   
            return_next7_without_previous_week_control  0.02034  0.00044   

                                                                        ci95  
quality     return_next7                                  [0.01391, 0.01558]  
            visit_days_next7                              [0.00938, 0.01412]  
            reverse_check_past6_visit_days              [-0.01779, -0.01293]  
            return_next7_controlling_clicks_today          [0.01353, 0.0152]  
            return_next7_without_previous_week_control    [0.01807, 0.01978]  
quality_all return_next7                                  [0.01408, 0.01574]  
            visit_days_next7                               [0.0052, 0.00989]  
            reverse_check_past6_visit_days              [-0.02727, -0.02248]  
            return_next7_controlling_clicks_today         [0.01357, 0.01523]  
            return_next7_without_previous_week_control     [0.01949, 0.0212]

In [13]:
# Joint model: return, visit days, reverse check, and return without last week's visits as a control
pd.concat({out: pd.DataFrame(res["B_joint"][out]).T
           for out in ("return_next7", "visit_days_next7", "reverse_check_past6_visit_days",
                       "return_next7_without_previous_week_control")}, axis=1)

return_next7                                 \
                             coef       se                  ci95   
quality_z                 0.01325  0.00044     [0.01239, 0.0141]   
video_z                   0.01079  0.00041       [0.01, 0.01159]   
popularity_z             -0.00715  0.00085  [-0.00882, -0.00548]   
age_z                     0.00421  0.00042    [0.00339, 0.00502]   
creator_variety_z         -0.0019  0.00037  [-0.00263, -0.00117]   
topic_variety_z          -0.00291  0.00043  [-0.00376, -0.00207]   
log_cards_today          -0.01057  0.00084  [-0.01221, -0.00893]   
log_prev_week_visits     -0.26496  0.00127  [-0.26745, -0.26246]   

                     visit_days_next7                                 \
                                 coef       se                  ci95   
quality_z                     0.01079  0.00125    [0.00835, 0.01323]   
video_z                       0.00768  0.00119    [0.00535, 0.01001]   
popularity_z                 -0.00913  0.00228  [-0.01361, -0.00466]   
age_z                         0.00519  0.00117    [0.00289, 0.00749]   
creator_variety_z            -0.00037  0.00116    [-0.00265, 0.0019]   
topic_variety_z               0.00078  0.00134   [-0.00185, 0.00341]   
log_cards_today               0.06134  0.00254    [0.05636, 0.06631]   
log_prev_week_visits         -0.85975  0.00352  [-0.86665, -0.85284]   

                     reverse_check_past6_visit_days           \
                                               coef       se   
quality_z                                   0.00206  0.00126   
video_z                                      0.0045  0.00121   
popularity_z                               -0.20774  0.00235   
age_z                                       0.00289  0.00119   
creator_variety_z                           0.00063  0.00118   
topic_variety_z                             0.01467  0.00138   
log_cards_today                              0.0919  0.00258   
log_prev_week_visits                            NaN      NaN   

                                            \
                                      ci95   
quality_z              [-0.00041, 0.00454]   
video_z                 [0.00213, 0.00686]   
popularity_z          [-0.21234, -0.20314]   
age_z                   [0.00054, 0.00523]   
creator_variety_z      [-0.00168, 0.00293]   
topic_variety_z         [0.01196, 0.01738]   
log_cards_today         [0.08684, 0.09696]   
log_prev_week_visits                   NaN   

                     return_next7_without_previous_week_control           \
                                                           coef       se   
quality_z                                                0.0151  0.00045   
video_z                                                 0.01289  0.00042   
popularity_z                                            0.01427  0.00088   
age_z                                                     0.005  0.00043   
creator_variety_z                                      -0.00254  0.00038   
topic_variety_z                                        -0.00525  0.00044   
log_cards_today                                        -0.02566  0.00087   
log_prev_week_visits                                        NaN      NaN   

                                            
                                      ci95  
quality_z               [0.01423, 0.01597]  
video_z                  [0.01207, 0.0137]  
popularity_z            [0.01255, 0.01598]  
age_z                   [0.00416, 0.00584]  
creator_variety_z     [-0.00329, -0.00178]  
topic_variety_z       [-0.00612, -0.00439]  
log_cards_today       [-0.02736, -0.02396]  
log_prev_week_visits                   NaN

## 7. Policy simulation

The cutoff is held at the lowest 10% of each day before any policy. The content levers use the best first page that can be built from the cards the user was shown that day.

In [14]:
# Inactive share under each policy, with the range over the coefficient intervals
S = pd.read_parquet(O / f"19_scores{TAG}.parquet")
G = pd.read_parquet(O / f"21_user_gains{TAG}.parquet").set_index("userId")
J = json.loads((O / f"23_content_mix{TAG}.json").read_text())["B_joint"]["return_next7"]   # one joint model for both levers
bq_, bv_ = J["quality_z"], J["video_z"]
cut = S.groupby("origin_day").expected_click_prob.transform(lambda s: s.quantile(0.10)).values
gain = S.userId.map(G.click_gain_platform_rate_creator_aware).fillna(1.0).values
pr, pc = S.p_return.values, S.p_click.values


def inactive(r, c):
    return float((np.clip(r, 0, 1) * np.clip(c, 0, 0.99) <= cut).mean())


base = inactive(pr, pc)

# section 6.2: the shift each test origin's first page could actually make, in the joint model's within-user SD units
SD = json.loads((O / f"23_content_mix{TAG}.json").read_text())["B_joint"]["within_user_sd"]
F = S[["userId", "origin_day"]].merge(FEAS, on=["userId", "origin_day"], how="left")
has_first = F.quality.notna().values
dq_q = np.nan_to_num(((F.q_bestq - F.quality) / SD["quality"]).values)   # best-quality page: quality gain
dv_q = np.nan_to_num(((F.v_bestq - F.video) / SD["video"]).values)        # ...and its side effect on video
dv_v = np.nan_to_num(((F.v_bestv - F.video) / SD["video"]).values)        # most-video page: video gain
dq_v = np.nan_to_num(((F.q_bestv - F.quality) / SD["quality"]).values)    # ...and its side effect on quality
assert (dq_q >= -1e-9).all() and (dv_v >= -1e-9).all()                  # the best page can never be worse
# the better of the two rebuilt pages for each origin, judged by its predicted gain in return (point estimates)
pick_q = (bq_["coef"] * dq_q + bv_["coef"] * dv_q) >= (bq_["coef"] * dq_v + bv_["coef"] * dv_v)
dq_b, dv_b = np.where(pick_q, dq_q, dq_v), np.where(pick_q, dv_q, dv_v)


def policy(beta_q=0.0, shift_q=0.0, beta_v=0.0, shift_v=0.0, rerank=False, mask=None):
    m = np.ones(len(S), bool) if mask is None else mask
    r = pr + m * (beta_q * shift_q + beta_v * shift_v)
    c = pc * np.where(m & rerank, gain, 1.0)
    return inactive(r, c)


POLICIES = {"A_rerank_pages": dict(rerank=True),
            # achievable versions: each origin's own best first page from the cards it was shown that day
            "B_feasible_best_quality_first_page": dict(shift_q=dq_q, shift_v=dv_q),
            "C_feasible_most_video_first_page": dict(shift_q=dq_v, shift_v=dv_v),
            "A+B_feasible": dict(rerank=True, shift_q=dq_q, shift_v=dv_q),
            "best_feasible_first_page": dict(shift_q=dq_b, shift_v=dv_b),
            "A+best_feasible_first_page": dict(rerank=True, shift_q=dq_b, shift_v=dv_b)}
res = {"test_origins": len(S), "inactive_share_baseline": round(base, 4), "policies": {}}
for name, kw in POLICIES.items():
    out = {"point": round(policy(beta_q=bq_["coef"], beta_v=bv_["coef"], **kw), 4)}
    # every corner of the two intervals: the shifts in quality and video can have opposite signs
    ends = [policy(beta_q=bq_["ci95"][i], beta_v=bv_["ci95"][j], **kw) for i in (0, 1) for j in (0, 1)]
    out["low"], out["high"] = round(min(ends), 4), round(max(ends), 4)     # range over the coefficients' 95% intervals
    out["inactive_reduction_pct_of_baseline_point"] = round(100 * (1 - out["point"] / base), 1)
    res["policies"][name] = out

# who moves? among origins inactive at baseline, the share each policy lifts above the cutoff, by baseline band
rank = pd.Series(S.expected_click_prob.values).groupby(S.origin_day.values).rank(pct=True).values


def lifted(beta_q=0.0, shift_q=0.0, beta_v=0.0, shift_v=0.0, rerank=False, mask=None):
    r = np.clip(pr + beta_q * shift_q + beta_v * shift_v, 0, 1)
    c = np.clip(pc * (gain if rerank else 1.0), 0, 0.99)
    return r * c > cut


was = S.expected_click_prob.values <= cut
res["share_of_inactive_lifted_by_baseline_band"] = {}
for name in ("A_rerank_pages", "best_feasible_first_page",
             "A+best_feasible_first_page"):
    up = lifted(beta_q=bq_["coef"], beta_v=bv_["coef"], **POLICIES[name])
    res["share_of_inactive_lifted_by_baseline_band"][name] = {
        f"rank_{lo:.0%}-{lo + 0.02:.0%}": round(float(up[was & (rank > lo) & (rank <= lo + 0.02)].mean()), 4)
        for lo in np.arange(0, 0.10, 0.02)}
res["band_note"] = "bands = position in the baseline ranking of the same day (0-2% = least active); cutoff = 10%"


def dist(x):
    x = x[has_first]
    return {"mean": round(float(x.mean()), 3), "median": round(float(np.median(x)), 3),
            **{f"p{q}": round(float(np.quantile(x, q / 100)), 3) for q in (25, 75, 90)},
            "share_zero": round(float((x <= 1e-9).mean()), 4), "share_ge_0.5sd": round(float((x >= 0.5).mean()), 4),
            "share_ge_1sd": round(float((x >= 1.0).mean()), 4)}


def hist(x):
    """Share of test origins with a first page by achievable shift: exactly zero, then bands of 0.25 SD, 4 or more."""
    x = x[has_first]
    edges = np.arange(0, 4.01, 0.25)
    out = {"0": round(float((x <= 1e-9).mean()), 4)}
    for lo, hi in zip(edges[:-1], edges[1:]):
        out[f"{lo:.2f}-{hi:.2f}"] = round(float(((x > max(lo, 1e-9)) & (x <= hi)).mean()), 4)
    out["4.00+"] = round(float((x > 4).mean()), 4)
    return out


res["feasibility"] = {
    "test_origins_with_first_page": int(has_first.sum()), "test_origins": len(S),
    "within_user_sd": {"quality": SD["quality"], "video": SD["video"]},
    "best_quality_first_page": {"quality_shift_sd": dist(dq_q), "video_side_effect_sd": dist(dv_q)},
    "most_video_first_page": {"video_shift_sd": dist(dv_v), "quality_side_effect_sd": dist(dq_v)},
    "histogram": {"quality_shift_best_quality_page": hist(dq_q), "video_shift_most_video_page": hist(dv_v)},
    "note": ("first page rebuilt from the cards the user was shown that day, same number of cards; origins without a "
             "first page shift by 0; shifts in the within-user SD units of the joint model")}
# a single page that day: the first page can only be rebuilt from cards shown later the same day
single = (F.pages.values == 1) & has_first
res["feasibility"]["share_with_a_single_page"] = round(float(single.sum() / has_first.sum()), 4)
res["feasibility"]["share_single_page_among_unchangeable_quality"] = round(
    float((single & (dq_q <= 1e-9)).sum() / (has_first & (dq_q <= 1e-9)).sum()), 4)

# the click side of the first-page levers: a better first page also raises expected clicks (exposure model), applied to
# click propensity as re-ordering is; with re-ordering, the two gains multiply
FPG_ = pd.read_parquet(O / f"23_first_page_click_gains{TAG}.parquet").set_index("userId")
gq = S.userId.map(FPG_.gain_best_quality).fillna(1.0).values
gv = S.userId.map(FPG_.gain_most_video).fillna(1.0).values
gb = np.where(pick_q, gq, gv)


def policy_fp(fp, beta_q=0.0, shift_q=0.0, beta_v=0.0, shift_v=0.0, rerank=False):
    return inactive(pr + beta_q * shift_q + beta_v * shift_v, pc * fp * (gain if rerank else 1.0))


FP = {"B_feasible_best_quality_first_page": (gq, dict(shift_q=dq_q, shift_v=dv_q)),
      "C_feasible_most_video_first_page": (gv, dict(shift_q=dq_v, shift_v=dv_v)),
      "best_feasible_first_page": (gb, dict(shift_q=dq_b, shift_v=dv_b)),
      "A+best_feasible_first_page": (gb, dict(rerank=True, shift_q=dq_b, shift_v=dv_b))}
res["policies_with_first_page_clicks"] = {}
for name, (fp, kw) in FP.items():
    v = policy_fp(fp, beta_q=bq_["coef"], beta_v=bv_["coef"], **kw)
    res["policies_with_first_page_clicks"][name] = {"point": round(v, 4),
                                                    "inactive_reduction_pct_of_baseline_point": round(100 * (1 - v / base), 1)}
res["first_page_click_gain_per_user"] = {k: {"mean": round(float(FPG_[c].mean()), 4),
                                             "median": round(float(FPG_[c].median()), 4),
                                             "share_above_1": round(float((FPG_[c] > 1 + 1e-9).mean()), 4)}
                                         for k, c in (("best_quality", "gain_best_quality"), ("most_video", "gain_most_video"))}
res["assumptions"] = ("cutoff fixed at the baseline bottom-10% per day; section-5 associations read as effects; "
                      "effects additive; re-ordering leaves return unchanged; the *_feasible policies use each origin's "
                      "achievable first page")
(O / f"24_policy{TAG}.json").write_text(json.dumps(res, indent=2))
pd.DataFrame(res["policies"]).T.assign(baseline=res["inactive_share_baseline"])

,point,low,high,inactive_reduction_pct_of_baseline_point,baseline
A_rerank_pages,0.0878,0.0878,0.0878,12.2,0.1
B_feasible_best_quality_first_page,0.0994,0.0994,0.0995,0.6,0.1
C_feasible_most_video_first_page,0.0993,0.0993,0.0993,0.7,0.1
A+B_feasible,0.0874,0.0874,0.0874,12.6,0.1
best_feasible_first_page,0.0993,0.0993,0.0993,0.7,0.1
A+best_feasible_first_page,0.0872,0.0872,0.0872,12.8,0.1


In [15]:
# Share of baseline-inactive snapshots lifted above the cutoff, by baseline band
pd.DataFrame(res["share_of_inactive_lifted_by_baseline_band"])

,A_rerank_pages,best_feasible_first_page,A+best_feasible_first_page
rank_0%-2%,0.0000,0.0002,0.0004
rank_2%-4%,0.0044,0.0011,0.0059
rank_4%-6%,0.0261,0.0050,0.0317
rank_6%-8%,0.1211,0.0074,0.1294
rank_8%-10%,0.5140,0.0232,0.5287


In [16]:
# Achievable shifts in within-user SD units (test origins with a first page)
pd.concat({k: pd.DataFrame(v).T for k, v in res["feasibility"].items()
           if isinstance(v, dict) and k not in ("within_user_sd", "histogram")})

mean  median  p25  p75  \
best_quality_first_page quality_shift_sd        0.359     0.0  0.0  0.0   
                        video_side_effect_sd   -0.006     0.0  0.0  0.0   
most_video_first_page   video_shift_sd          0.279     0.0  0.0  0.0   
                        quality_side_effect_sd  0.237     0.0  0.0  0.0   

                                                  p90  share_zero  \
best_quality_first_page quality_shift_sd        1.581      0.8256   
                        video_side_effect_sd    0.000      0.9457   
most_video_first_page   video_shift_sd          1.135      0.8647   
                        quality_side_effect_sd  0.938      0.8402   

                                                share_ge_0.5sd  share_ge_1sd  
best_quality_first_page quality_shift_sd                0.1580        0.1316  
                        video_side_effect_sd            0.0464        0.0259  
most_video_first_page   video_shift_sd                  0.1281        0.1023  
                        quality_side_effect_sd          0.1299        0.0960

In [17]:
# Inactive share and relative reduction under each policy, at four cutoffs
sens = {}
for q_cut in (0.05, 0.10, 0.20, 0.25):
    cut = S.groupby("origin_day").expected_click_prob.transform(lambda s: s.quantile(q_cut)).values
    base_q = inactive(pr, pc)                                           # inactive() reads the current cut
    row = {"baseline": round(base_q, 4)}
    for name, kw in POLICIES.items():
        v = policy(beta_q=bq_["coef"], beta_v=bv_["coef"], **kw)
        row[name] = round(v, 4)
        row[name + "_reduction_pct"] = round(100 * (1 - v / base_q), 1)
    sens[f"bottom_{int(q_cut * 100)}pct"] = row
(O / f"24_policy_cutoffs{TAG}.json").write_text(json.dumps(sens, indent=2))
table = pd.DataFrame(sens)
table.loc[[k for k in table.index if k.endswith("_reduction_pct")]].rename(index=lambda k: k[:-14])

,bottom_5pct,bottom_10pct,bottom_20pct,bottom_25pct
A_rerank_pages,13.6,12.2,10.7,10.3
B_feasible_best_quality_first_page,0.4,0.6,0.7,0.7
C_feasible_most_video_first_page,0.5,0.7,0.9,1.0
A+B_feasible,13.9,12.6,11.3,11.0
best_feasible_first_page,0.5,0.7,0.9,1.0
A+best_feasible_first_page,14.0,12.8,11.5,11.2


## 8. Value and test size

The gains at the scale of the whole platform, and the number of users a test would need to detect them.

In [18]:
# Platform scale, extra clicks and the users a randomized test would need (two arms, 5% significance, 80% power)
from scipy.stats import norm
con = connect()
plat = con.execute(f"""SELECT CAST(dt AS INT) AS dt, sum(CAST(userImprssionCount AS BIGINT)) AS impr,
                              sum(CAST(userClickCount AS BIGINT)) AS clicks
                       FROM '{P}/mlog_stats.parquet' GROUP BY 1 ORDER BY 1""").df()
samp = con.execute(f"""SELECT CAST(dt AS INT) AS dt, count(*) AS impr, sum(isClick) AS clicks
                       FROM read_parquet('{P}/impressions_place/d*.parquet') GROUP BY 1 ORDER BY 1""").df()
con.close()
pw = plat[plat.dt.between(16, 30)]
sw = samp[samp.dt.between(16, 30)]
z = norm.ppf(0.975) + norm.ppf(0.80)
R21 = json.loads((O / f"21_observed_rerank{TAG}.json").read_text())
R24 = json.loads((O / f"24_policy{TAG}.json").read_text())
R21m = json.loads((O / f"21_rerank{TAG}.json").read_text())
cl = R21["test_user_clicks_days16_30"]


def n_clicks(rel_gain):
    """Users per arm to detect a relative change in clicks per user over 15 days (normal approximation)."""
    return int(np.ceil(2 * z ** 2 * cl["sd"] ** 2 / (rel_gain * cl["mean"]) ** 2))


def n_share(p0, p1):
    return int(np.ceil(z ** 2 * (p0 * (1 - p0) + p1 * (1 - p1)) / (p0 - p1) ** 2))


p0 = R24["inactive_share_baseline"]
biz = {"platform_daily_clicks_days16_30": int(pw.clicks.mean()), "platform_daily_impressions_days16_30": int(pw.impr.mean()),
       "sample_share_of_platform_impressions": round(float(sw.impr.sum() / pw.impr.sum()), 4),
       "extra_daily_clicks": {name: int(R21["all_users"][name]["gain_pct"] / 100 * pw.clicks.mean())
                              for name in ("platform_rate", "platform_rate_creator_aware")},
       "users_per_arm": {
           "clicks_observed_gain_creator_rule": n_clicks(R21["test_users"]["platform_rate_creator_aware"]["gain_pct"] / 100),
           "clicks_model_gain_creator_rule": n_clicks(R21m["rerank_within_pages"]["expected_clicks_change_pct_platform_rate_creator_aware"] / 100)},
       # not a number of users: snapshots treated as independent, and the simulated drop comes from the model (section 4)
       "user_days_per_arm_iid_not_used": {
           "inactive_share_rerank": n_share(p0, R24["policies"]["A_rerank_pages"]["point"]),
           "inactive_share_best_feasible_first_page": n_share(p0, R24["policies"]["best_feasible_first_page"]["point"]),
           "inactive_share_rerank_plus_first_page": n_share(p0, R24["policies"]["A+best_feasible_first_page"]["point"])},
       "note": ("users per arm from clicks per test user over days 16-30; the inactive-share sizes count user-days as independent "
                "platform counts from mlog_stats (all users of the feed)")}
(O / f"27_business_case{TAG}.json").write_text(json.dumps(biz, indent=2))
pd.Series({k: v for k, v in biz.items() if not isinstance(v, dict)}, name="value").to_frame()

,value
platform_daily_clicks_days16_30,1966445
platform_daily_impressions_days16_30,40061028
sample_share_of_platform_impressions,0.0482
note,users per arm from clicks per test user over d...


## 9. Summary

What a user is shown carries information the user's history does not. The slot shapes clicks, and re-ordering raises clicks both in the model and in the logged data. First pages are limited by the cards available, and the levers reach users near the cutoff.